# AI Personal Finance Assistant — Complete Agentic RAG Multi-Agent System
### Final Project: Developing a Multi-Agent AI Agent System

This single Kaggle notebook implements a professional end-to-end educational personal-finance assistant using PDF RAG, hybrid retrieval, reranking, intent routing, multiple specialized agents, short/long-term memory, deterministic tools, workflow orchestration, automated reports, evaluation, observability, visualizations, and an optional Gradio interface.

The project is intentionally **not** a simple Chat-with-PDF chatbot.


## 01–04. Project Overview, Problem, Objectives, Architecture

**Target workflow**

`USER → Input Validation → Memory Recall → Intent Router → Agent Selection → RAG / Tools / Specialist Agents → Coordinator → Memory Update → Sources → Final Answer → Automated Outputs`

**Expected architecture**

- Router Agent
- Research/RAG Agent
- Financial Education Agent
- Budget Agent
- Debt Agent
- Investment Education Agent
- Summary Agent
- Calculator/Tools Agent
- Memory Agent
- Response Coordinator

Specialists may execute in parallel; synthesis occurs after their outputs are available.


In [ ]:
# 05. Environment, reproducibility, and GPU detection
import os, sys, platform, json, re, math, time, random, sqlite3, logging, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

DEVICE = "cuda" if TORCH_AVAILABLE and torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"

print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)
print("Python :", sys.version.split()[0])
print("PyTorch:", getattr(torch, "__version__", "not installed"))
print("CUDA   :", getattr(torch.version, "cuda", None) if TORCH_AVAILABLE else None)
print("Device :", GPU_NAME)
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("VRAM   :", round(props.total_memory / (1024**3), 2), "GB")


ENVIRONMENT
Python : 3.13.15
PyTorch: 2.11.0+cu128
CUDA   : 12.8
Device : Tesla T4
VRAM   : 14.56 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 06. Dependencies
import subprocess, sys

PACKAGES = [
    "pymupdf",
    "sentence-transformers",
    "faiss-cpu",
    "rank-bm25",
    "scikit-learn",
    "groq",
    "openai",
    "google-generativeai",
    "gradio"
]
for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("Dependency installation attempted. Optional components degrade gracefully.")


## 07–08. Configuration and API Secrets

Set `PDF_PATH` to the real textbook path. No API key is hardcoded. Kaggle Secrets are read through `UserSecretsClient`, with environment variables as a fallback.

Recommended secret names:

- `GROQ_API_KEY`
- `OPENAI_API_KEY`
- `GOOGLE_API_KEY`

`LLM_PROVIDER` can be `groq`, `openai`, or `gemini`.


In [ ]:
# ============================================================
# ⚙️ CONFIGURATION — GOOGLE COLAB
# ============================================================

from pathlib import Path
import os

BASE_DIR = Path("/content")

OUTPUT_DIR = BASE_DIR / "outputs"
ARTIFACT_DIR = BASE_DIR / "artifacts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 📄 PDF
# ============================================================

PDF_PATH = "https://drive.google.com/file/d/1lnOJdbCQhJQi2LnRs2F1UFniSq1ogLX-/view?usp=sharing"


# ============================================================
# 🤖 LLM CONFIGURATION
# ============================================================

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()

LLM_MODEL = os.getenv("LLM_MODEL", "") or {
    "groq": "openai/gpt-oss-20b",
    "openai": "gpt-4o-mini",
    "gemini": "gemini-3.6-flash"
}.get(LLM_PROVIDER, "")


# ============================================================
# 🔎 RETRIEVAL CONFIGURATION
# ============================================================

EMBEDDING_MODEL = os.getenv(
    "EMBEDDING_MODEL",
    "BAAI/bge-small-en-v1.5"
)

RERANKER_MODEL = os.getenv(
    "RERANKER_MODEL",
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


# ============================================================
# 🎯 RETRIEVAL PARAMETERS
# ============================================================

TOP_K_SEMANTIC = 12
TOP_K_BM25 = 12
TOP_K_FUSED = 10
TOP_K_RERANK = 5


# ============================================================
# 📦 CHUNKING
# ============================================================

CHUNK_SIZE = 920
CHUNK_OVERLAP = 120


# ============================================================
# 🧠 CONTEXT / GROUNDING
# ============================================================

MAX_CONTEXT_CHARS = 12000
MIN_RETRIEVAL_SCORE = 0.20


# ============================================================
# 🔐 API KEY
# ============================================================

def load_secret(name):

    # Environment variable
    value = os.getenv(name)

    if value:
        return value

    # Google Colab Secrets
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


ENV_KEY = {
    "groq": "GROQ_API_KEY",
    "openai": "OPENAI_API_KEY",
    "gemini": "GOOGLE_API_KEY"
}

API_KEY = load_secret(
    ENV_KEY.get(LLM_PROVIDER, "")
)


# ============================================================
# ✅ CONFIGURATION CHECK
# ============================================================

print("=" * 70)
print("⚙️ PERSONAL FINANCE AGENTIC RAG — CONFIGURATION")
print("=" * 70)

print("LLM_PROVIDER   :", LLM_PROVIDER)
print("LLM_MODEL      :", LLM_MODEL)
print("PDF_PATH       :", PDF_PATH)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("RERANKER_MODEL :", RERANKER_MODEL)

print(
    "PDF exists     :",
    Path(PDF_PATH).exists()
)

print(
    "API configured :",
    bool(API_KEY)
)

print("=" * 70)

## 09–13. PDF Validation, Ingestion, Cleaning, Chunking, Metadata

The source PDF is processed at page level. Every chunk keeps page, chapter, section, source, and chunk ID so citations are derived from actual metadata rather than invented.


In [ ]:
# ============================================================
# 📄 PDF INGESTION — BUILD `pages`
# Google Colab
# ============================================================

from pathlib import Path
from dataclasses import dataclass
import pymupdf


@dataclass
class PageRecord:
    page: int
    source: str
    text: str


# ============================================================
# 🔎 FIND PDF
# ============================================================

pdf_file = Path(PDF_PATH)

if not pdf_file.exists():

    print("⚠️ PDF_PATH not found:")
    print(pdf_file)

    # Search /content for PDF files
    pdf_candidates = list(Path("/content").rglob("*.pdf"))

    if not pdf_candidates:
        raise FileNotFoundError(
            "❌ No PDF file found anywhere under /content.\n"
            "Upload the Personal Finance PDF to Colab first."
        )

    print("\n📚 PDF files found:")

    for i, p in enumerate(pdf_candidates, 1):
        print(f"{i}. {p}")

    # Prefer Personal Finance.pdf
    preferred = [
        p for p in pdf_candidates
        if p.name.lower() == "personal finance.pdf"
    ]

    if preferred:
        pdf_file = preferred[0]
    else:
        # Otherwise use the first PDF found
        pdf_file = pdf_candidates[0]

    print("\n✅ Using PDF:")
    print(pdf_file)


# ============================================================
# 📖 EXTRACT PAGES
# ============================================================

pages = []

with pymupdf.open(pdf_file) as doc:

    for page_num, page in enumerate(doc, start=1):

        text = page.get_text("text").strip()

        if text:

            pages.append(
                PageRecord(
                    page=page_num,
                    source=pdf_file.name,
                    text=text
                )
            )


# ============================================================
# ✅ VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("📄 PDF INGESTION COMPLETE")
print("=" * 70)

print(f"PDF    : {pdf_file}")
print(f"Pages  : {len(pages):,}")

if not pages:
    raise RuntimeError(
        "❌ No text was extracted from the PDF."
    )

print("\n📋 Sample:")
print("-" * 70)
print(pages[0].text[:500])

print("=" * 70)
print("✅ `pages` is ready for chunking.")
print("=" * 70)

⚠️ PDF_PATH not found:
https:/drive.google.com/file/d/1lnOJdbCQhJQi2LnRs2F1UFniSq1ogLX-/view?usp=sharing

📚 PDF files found:
1. /content/drive/MyDrive/Fundamentals-of-Finance-1721351926.pdf
2. /content/drive/MyDrive/Fareed Anwar.pdf
3. /content/drive/MyDrive/Fareed-ali-mohey-eldeen-anwar-FlowCV-Resume-20241012 (1) (2).pdf
4. /content/drive/MyDrive/Fareed-ali-mohey-eldeen-anwar-FlowCV-Resume-20241012 (1) (1).pdf
5. /content/drive/MyDrive/Fareed-ali-mohey-eldeen-anwar-FlowCV-Resume-20241012 (1).pdf
6. /content/drive/MyDrive/D hub   fareed.pdf
7. /content/drive/MyDrive/Fareed-ali-mohey-eldeen-anwar-FlowCV-Resume-20241219 (1).pdf
8. /content/drive/MyDrive/Fareed-ali-mohey-eldeen-anwar-FlowCV-Resume-20241219.pdf
9. /content/drive/MyDrive/Fareed-ali-mohey-eldeen-anwar-FlowCV-Resume-20250204.pdf
10. /content/drive/MyDrive/Fareed Ali Mohey Eldin Anwar (4).pdf
11. /content/drive/MyDrive/Fareed Ali Mohey Eldin Anwar (3).pdf
12. /content/drive/MyDrive/Fareed Ali Mohey Eldin Anwar (2).pdf
13. /con

In [ ]:
# ============================================================
# 📦 CHUNKING PIPELINE
# Simplified metadata: chunk_id, source, page, text
# ============================================================

class ChunkManager:
    def __init__(self, size=920, overlap=120):
        self.size = size
        self.overlap = overlap

    def split(self, text):
        if len(text) <= self.size:
            return [text]

        out = []
        start = 0

        while start < len(text):
            end = min(len(text), start + self.size)

            # Try to end the chunk at a sentence boundary
            if end < len(text):
                cut = text.rfind(". ", start, end)

                if cut > start + int(self.size * 0.55):
                    end = cut + 1

            part = text[start:end].strip()

            if part:
                out.append(part)

            if end >= len(text):
                break

            start = max(
                end - self.overlap,
                start + 1
            )

        return out

    def build(self, pages):
        rows = []

        for p in pages:
            page_chunks = self.split(p.text)

            for j, text in enumerate(page_chunks, 1):
                rows.append({
                    "chunk_id": f"p{p.page:04d}_c{j:02d}",
                    "source": p.source,
                    "page": p.page,
                    "text": text
                })

        return rows


# ============================================================
# 🚀 BUILD CHUNKS
# ============================================================

chunk_manager = ChunkManager(
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

chunks = chunk_manager.build(pages)

chunks_df = pd.DataFrame(chunks)


# ============================================================
# 📊 VALIDATION
# ============================================================

print("=" * 70)
print("📦 PDF CHUNKING COMPLETE")
print("=" * 70)

print(f"📄 Pages       : {len(pages):,}")
print(f"🧩 Chunks      : {len(chunks):,}")
print(f"📏 Chunk size  : {CHUNK_SIZE}")
print(f"🔄 Overlap     : {CHUNK_OVERLAP}")

print("\n📋 Columns:")
print(list(chunks_df.columns))

print("\n🔍 Sample:")
display(chunks_df.head())

print("\n📊 Chunks per page:")
print(chunks_df.groupby("page").size().describe())

print("=" * 70)
print("✅ Chapter/Section extraction removed.")
print("✅ Metadata simplified.")
print("✅ Ready for embedding + vector store.")
print("=" * 70)

## 14–17. Embeddings, FAISS, Hybrid Retrieval, Reranking

The baseline uses `BAAI/bge-small-en-v1.5`, a practical sentence-transformers model for Kaggle. FAISS handles semantic retrieval; BM25 adds keyword matching; a cross-encoder reranks the combined candidates when available.


In [ ]:
# ============================================================
# 🔎 RETRIEVAL STACK
# Simplified metadata:
# chunk_id, source, page, text
# ============================================================
import  numpy as np

from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi


# ============================================================
# 🧠 EMBEDDING MANAGER
# ============================================================

class EmbeddingManager:

    def __init__(self, name, device):
        self.name = name
        self.model = SentenceTransformer(
            name,
            device=device
        )

    def encode(self, texts, batch_size=64):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")


# ============================================================
# 🗂️ VECTOR STORE — FAISS
# ============================================================

class VectorStore:

    def __init__(self):
        self.index = None

    def build(self, matrix):

        if matrix.size == 0:
            return

        self.index = faiss.IndexFlatIP(
            matrix.shape[1]
        )

        self.index.add(matrix)

    def search(self, qv, k):

        if (
            self.index is None
            or self.index.ntotal == 0
        ):
            return (
                np.empty((1, 0)),
                np.empty((1, 0), dtype=int)
            )

        k = min(
            k,
            self.index.ntotal
        )

        return self.index.search(
            qv.reshape(1, -1).astype("float32"),
            k
        )


# ============================================================
# 🔤 TOKENIZER — BM25
# ============================================================

def tokenize(text):

    return re.findall(
        r"[A-Za-z0-9$%]+",
        str(text).lower()
    )


# ============================================================
# 🚀 BUILD EMBEDDING + VECTOR + RERANKER STACK
# ============================================================

try:

    embedder = EmbeddingManager(
        EMBEDDING_MODEL,
        DEVICE
    )

    embeddings = (
        embedder.encode(
            [c["text"] for c in chunks]
        )
        if chunks
        else np.empty(
            (0, 384),
            dtype="float32"
        )
    )

    vector_store = VectorStore()

    vector_store.build(
        embeddings
    )

    # --------------------------------------------------------
    # Cross Encoder Reranker
    # --------------------------------------------------------

    reranker_enabled = False
    cross_encoder = None

    try:

        from sentence_transformers import CrossEncoder

        cross_encoder = CrossEncoder(
            RERANKER_MODEL,
            device=DEVICE
        )

        reranker_enabled = True

    except Exception as e:

        cross_encoder = None

        print(
            "Reranker unavailable:",
            type(e).__name__
        )

except Exception as e:

    embedder = None

    embeddings = np.empty(
        (0, 384),
        dtype="float32"
    )

    vector_store = VectorStore()

    cross_encoder = None

    reranker_enabled = False

    print(
        "Embedding stack unavailable:",
        repr(e)
    )


# ============================================================
# 🔀 HYBRID RETRIEVER
# ============================================================

class HybridRetriever:

    def __init__(
        self,
        rows,
        embedder,
        store
    ):

        self.rows = rows
        self.embedder = embedder
        self.store = store

        self.bm25 = (
            BM25Okapi(
                [
                    tokenize(r["text"])
                    for r in rows
                ]
            )
            if rows
            else None
        )

    @staticmethod
    def minmax(a):

        if len(a) == 0:
            return a

        lo = float(a.min())
        hi = float(a.max())

        if hi - lo < 1e-9:
            return np.zeros_like(a)

        return (
            (a - lo) /
            (hi - lo)
        )

    def retrieve(
        self,
        query,
        k=TOP_K_FUSED
    ):

        if (
            not self.rows
            or self.embedder is None
        ):
            return []

        # ----------------------------------------------------
        # Semantic Search
        # ----------------------------------------------------

        qv = self.embedder.encode(
            [query],
            batch_size=1,
            show_progress_bar=True
        )[0]

        s, ids = self.store.search(
            qv,
            TOP_K_SEMANTIC
        )

        semantic = {
            int(i): float(sc)
            for sc, i in zip(
                s[0],
                ids[0]
            )
            if i >= 0
        }

        # ----------------------------------------------------
        # BM25 Keyword Search
        # ----------------------------------------------------

        keyword = {}

        if self.bm25:

            raw = np.asarray(
                self.bm25.get_scores(
                    tokenize(query)
                ),
                float
            )

            for idx in np.argsort(raw)[::-1][
                :TOP_K_BM25
            ]:

                keyword[int(idx)] = float(
                    raw[idx]
                )

        # ----------------------------------------------------
        # Hybrid Fusion
        # ----------------------------------------------------

        all_ids = sorted(
            set(semantic) |
            set(keyword)
        )

        if not all_ids:
            return []

        sa = self.minmax(
            np.array(
                [
                    semantic.get(i, 0)
                    for i in all_ids
                ]
            )
        )

        ka = self.minmax(
            np.array(
                [
                    keyword.get(i, 0)
                    for i in all_ids
                ]
            )
        )

        fused = (
            0.65 * sa +
            0.35 * ka
        )

        order = np.argsort(
            fused
        )[::-1][:k]

        out = []

        for pos in order:

            idx = all_ids[int(pos)]

            # Copy original chunk metadata
            r = dict(
                self.rows[idx]
            )

            # Add retrieval scores only
            r.update({

                "semantic_score":
                    semantic.get(idx, 0),

                "bm25_score":
                    keyword.get(idx, 0),

                "fused_score":
                    float(
                        fused[int(pos)]
                    ),

                "_row_index":
                    idx
            })

            out.append(r)

        return out


# ============================================================
# 🔎 CREATE HYBRID RETRIEVER
# ============================================================

retriever = HybridRetriever(
    chunks,
    embedder,
    vector_store
)


# ============================================================
# 🎯 RERANKER
# ============================================================

class Reranker:

    def rerank(
        self,
        query,
        candidates,
        k=TOP_K_RERANK
    ):

        if not candidates:
            return []

        # ----------------------------------------------------
        # No Cross Encoder
        # ----------------------------------------------------

        if not reranker_enabled:

            return sorted(
                candidates,
                key=lambda x:
                    x["fused_score"],
                reverse=True
            )[:k]

        # ----------------------------------------------------
        # Cross Encoder
        # ----------------------------------------------------

        pairs = [
            (
                query,
                c["text"]
            )
            for c in candidates
        ]

        scores = np.asarray(
            cross_encoder.predict(
                pairs
            ),
            float
        )

        order = np.argsort(
            scores
        )[::-1][:k]

        out = []

        for idx in order:

            r = dict(
                candidates[int(idx)]
            )

            r["rerank_score"] = float(
                scores[int(idx)]
            )

            out.append(r)

        return out


# ============================================================
# 🚀 CREATE RERANKER
# ============================================================

reranker = Reranker()


# ============================================================
# 📚 RAG ENGINE
# ============================================================

class RAGEngine:

    # --------------------------------------------------------
    # Search
    # --------------------------------------------------------

    def search(self, query):

        candidates = retriever.retrieve(
            query
        )

        ranked = reranker.rerank(
            query,
            candidates
        )

        grounded = (
            bool(ranked)
            and ranked[0].get(
                "fused_score",
                0
            ) >= MIN_RETRIEVAL_SCORE
        )

        ranked = [
            r
            for r in ranked
            if r.get(
                "fused_score",
                0
            ) >= MIN_RETRIEVAL_SCORE
        ]

        return {
            "candidates": candidates,
            "reranked": ranked,
            "grounded": grounded
        }

    # --------------------------------------------------------
    # Context
    # --------------------------------------------------------

    def context(self, results):

        blocks = []
        total = 0

        for r in results:

            # IMPORTANT:
            # Only use metadata that actually exists.
            #
            # Available metadata:
            # chunk_id
            # source
            # page
            # text

            b = (
                f"[{r['source']} | "
                f"Page: {r['page']} | "
                f"Chunk: {r['chunk_id']}]\n"
                f"{r['text']}"
            )

            if (
                total + len(b)
                > MAX_CONTEXT_CHARS
            ):
                break

            blocks.append(b)
            total += len(b)

        return "\n\n".join(
            blocks
        )

    # --------------------------------------------------------
    # Citations
    # --------------------------------------------------------

    def citations(self, results):

        return [
            {
                "source": r["source"],
                "page": r["page"],
                "chunk_id": r["chunk_id"]
            }
            for r in results
        ]


# ============================================================
# 🚀 CREATE RAG ENGINE
# ============================================================

rag_engine = RAGEngine()


# ============================================================
# ✅ VALIDATION
# ============================================================

print("=" * 70)
print("🔎 RETRIEVAL STACK READY")
print("=" * 70)

print(
    "Embeddings:",
    embeddings.shape
)

print(
    "FAISS:",
    vector_store.index.ntotal
    if vector_store.index
    else 0
)

print(
    "Reranker:",
    reranker_enabled
)

print(
    "Metadata:",
    ["chunk_id", "source", "page", "text"]
)

print("=" * 70)
print("✅ Chapter/Section dependencies removed.")
print("✅ Hybrid retrieval ready.")
print("✅ Reranking ready.")
print("✅ RAG context/citations fixed.")
print("=" * 70)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (13 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

🔎 RETRIEVAL STACK READY
Embeddings: (273, 384)
FAISS: 273
Reranker: True
Metadata: ['chunk_id', 'source', 'page', 'text']
✅ Chapter/Section dependencies removed.
✅ Hybrid retrieval ready.
✅ Reranking ready.
✅ RAG context/citations fixed.


## 18–19. LLM Abstraction and Intent Router

`LLMClient` isolates provider-specific code. The Router asks the LLM for a structured JSON intent with a safe deterministic fallback if the provider is unavailable.


In [ ]:
# ============================================================
# 14–17. EMBEDDINGS, FAISS, HYBRID RETRIEVAL & RERANKING
# ============================================================

from sentence_transformers import SentenceTransformer
import faiss
from rank_bm25 import BM25Okapi


# ============================================================
# EMBEDDING MANAGER
# ============================================================

class EmbeddingManager:

    def __init__(self, name, device):
        self.name = name
        self.model = SentenceTransformer(
            name,
            device=device
        )

    def encode(self, texts, batch_size=64):

        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True
        ).astype("float32")


# ============================================================
# FAISS VECTOR STORE
# ============================================================

class VectorStore:

    def __init__(self):
        self.index = None

    def build(self, matrix):

        if matrix.size == 0:
            return

        self.index = faiss.IndexFlatIP(
            matrix.shape[1]
        )

        self.index.add(matrix)

    def search(self, qv, k):

        if (
            self.index is None
            or self.index.ntotal == 0
        ):
            return (
                np.empty((1, 0)),
                np.empty((1, 0), dtype=int)
            )

        k = min(
            k,
            self.index.ntotal
        )

        return self.index.search(
            qv.reshape(1, -1).astype("float32"),
            k
        )


# ============================================================
# TOKENIZER
# ============================================================

def tokenize(text):

    return re.findall(
        r"[A-Za-z0-9$%]+",
        text.lower()
    )


# ============================================================
# BUILD EMBEDDING + FAISS + RERANKER
# ============================================================

try:

    embedder = EmbeddingManager(
        EMBEDDING_MODEL,
        DEVICE
    )

    embeddings = (
        embedder.encode(
            [c["text"] for c in chunks]
        )
        if chunks
        else np.empty(
            (0, 384),
            dtype="float32"
        )
    )

    vector_store = VectorStore()

    vector_store.build(
        embeddings
    )

    reranker_enabled = False

    try:

        from sentence_transformers import CrossEncoder

        cross_encoder = CrossEncoder(
            RERANKER_MODEL,
            device=DEVICE
        )

        reranker_enabled = True

    except Exception as e:

        cross_encoder = None

        print(
            "Reranker unavailable:",
            type(e).__name__,
            e
        )

except Exception as e:

    embedder = None

    embeddings = np.empty(
        (0, 384),
        dtype="float32"
    )

    vector_store = VectorStore()

    cross_encoder = None

    reranker_enabled = False

    print(
        "Embedding stack unavailable:",
        repr(e)
    )


# ============================================================
# HYBRID RETRIEVER
# ============================================================

class HybridRetriever:

    def __init__(
        self,
        rows,
        embedder,
        store
    ):

        self.rows = rows
        self.embedder = embedder
        self.store = store

        self.bm25 = (
            BM25Okapi(
                [
                    tokenize(r["text"])
                    for r in rows
                ]
            )
            if rows
            else None
        )

    @staticmethod
    def minmax(a):

        if len(a) == 0:
            return a

        lo = float(a.min())
        hi = float(a.max())

        if hi - lo < 1e-9:
            return np.zeros_like(a)

        return (
            a - lo
        ) / (
            hi - lo
        )

    def retrieve(
        self,
        query,
        k=TOP_K_FUSED
    ):

        if (
            not self.rows
            or self.embedder is None
        ):
            return []

        qv = self.embedder.encode(
            [query],
            batch_size=1
        )[0]

        scores, ids = self.store.search(
            qv,
            TOP_K_SEMANTIC
        )

        semantic = {
            int(i): float(sc)
            for sc, i in zip(
                scores[0],
                ids[0]
            )
            if i >= 0
        }

        keyword = {}

        if self.bm25:

            raw = np.asarray(
                self.bm25.get_scores(
                    tokenize(query)
                ),
                float
            )

            for idx in np.argsort(raw)[::-1][
                :TOP_K_BM25
            ]:

                keyword[int(idx)] = float(
                    raw[idx]
                )

        all_ids = sorted(
            set(semantic)
            | set(keyword)
        )

        sa = self.minmax(
            np.array(
                [
                    semantic.get(i, 0)
                    for i in all_ids
                ]
            )
        )

        ka = self.minmax(
            np.array(
                [
                    keyword.get(i, 0)
                    for i in all_ids
                ]
            )
        )

        fused = (
            0.65 * sa
            + 0.35 * ka
        )

        order = np.argsort(
            fused
        )[::-1][:k]

        out = []

        for pos in order:

            idx = all_ids[int(pos)]

            r = dict(
                self.rows[idx]
            )

            r.update({

                "semantic_score":
                    semantic.get(idx, 0),

                "bm25_score":
                    keyword.get(idx, 0),

                "fused_score":
                    float(
                        fused[int(pos)]
                    ),

                "_row_index":
                    idx
            })

            out.append(r)

        return out


retriever = HybridRetriever(
    chunks,
    embedder,
    vector_store
)


# ============================================================
# RERANKER
# ============================================================

class Reranker:

    def rerank(
        self,
        query,
        candidates,
        k=TOP_K_RERANK
    ):

        if not candidates:
            return []

        if not reranker_enabled:

            return sorted(
                candidates,
                key=lambda x:
                    x["fused_score"],
                reverse=True
            )[:k]

        scores = np.asarray(
            cross_encoder.predict(
                [
                    (
                        query,
                        c["text"]
                    )
                    for c in candidates
                ]
            ),
            float
        )

        order = np.argsort(
            scores
        )[::-1][:k]

        out = []

        for idx in order:

            r = dict(
                candidates[int(idx)]
            )

            r["rerank_score"] = float(
                scores[int(idx)]
            )

            out.append(r)

        return out


reranker = Reranker()


# ============================================================
# RAG ENGINE
# ============================================================

class RAGEngine:

    def search(self, query):

        candidates = retriever.retrieve(
            query
        )

        ranked = reranker.rerank(
            query,
            candidates
        )

        grounded = (
            bool(ranked)
            and ranked[0].get(
                "fused_score",
                0
            ) >= MIN_RETRIEVAL_SCORE
        )

        ranked = [
            r
            for r in ranked
            if r.get(
                "fused_score",
                0
            ) >= MIN_RETRIEVAL_SCORE
        ]

        return {
            "candidates": candidates,
            "reranked": ranked,
            "grounded": grounded
        }


    # ========================================================
    # CONTEXT
    # ========================================================

    def context(
        self,
        results
    ):

        blocks = []
        total = 0

        for r in results:

            # Metadata schema:
            # chunk_id, source, page, text

            b = (
                f"[{r['source']} | "
                f"Page: {r['page']} | "
                f"Chunk: {r['chunk_id']}]\n"
                f"{r['text']}"
            )

            if (
                total + len(b)
                > MAX_CONTEXT_CHARS
            ):
                break

            blocks.append(b)
            total += len(b)

        return "\n\n".join(
            blocks
        )


    # ========================================================
    # CITATIONS
    # ========================================================

    def citations(
        self,
        results
    ):

        return [
            {
                "source": r["source"],
                "page": r["page"],
                "chunk_id": r["chunk_id"]
            }
            for r in results
        ]


# ============================================================
# CREATE RAG ENGINE
# ============================================================

rag_engine = RAGEngine()


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("🔎 RETRIEVAL STACK READY")
print("=" * 70)

print(
    "Embeddings:",
    embeddings.shape
)

print(
    "FAISS:",
    vector_store.index.ntotal
    if vector_store.index
    else 0
)

print(
    "Reranker:",
    reranker_enabled
)

print(
    "Metadata schema:",
    [
        "chunk_id",
        "source",
        "page",
        "text"
    ]
)

print(
    "Chapter dependency: ❌ Removed"
)

print(
    "Section dependency: ❌ Removed"
)

print("=" * 70)

In [ ]:
# ============================================================
# 🤖 LLM ABSTRACTION — GROQ / OPENAI / GEMINI
# ============================================================

INTENTS = [
    "budgeting",
    "saving",
    "debt",
    "credit",
    "investing",
    "retirement",
    "insurance",
    "taxes",
    "financial_planning",
    "financial_education",
    "book_summary",
    "comparison",
    "calculation",
    "general_question",
    "unsupported"
]


class LLMClient:

    def __init__(self, provider, api_key, model):

        self.provider = provider
        self.model = model

        if not api_key:
            raise ValueError(
                f"API key is missing for provider: {provider}"
            )

        # ----------------------------------------------------
        # GROQ
        # ----------------------------------------------------
        if provider == "groq":

            from groq import Groq

            self.client = Groq(
                api_key=api_key
            )

        # ----------------------------------------------------
        # OPENAI
        # ----------------------------------------------------
        elif provider == "openai":

            from openai import OpenAI

            self.client = OpenAI(
                api_key=api_key
            )

        # ----------------------------------------------------
        # GEMINI
        # ----------------------------------------------------
        elif provider == "gemini":

            from google import genai

            self.client = genai.Client(
                api_key=api_key
            )

        else:

            raise ValueError(
                "Provider must be 'groq', 'openai', or 'gemini'."
            )


    # ========================================================
    # TEXT GENERATION
    # ========================================================

    def text(self, system, user, temperature=0.15):

        # ----------------------------------------------------
        # GROQ
        # ----------------------------------------------------
        if self.provider == "groq":

            response = self.client.chat.completions.create(

                model=self.model,

                messages=[
                    {
                        "role": "system",
                        "content": system
                    },
                    {
                        "role": "user",
                        "content": user
                    }
                ],

                temperature=temperature
            )

            return response.choices[0].message.content


        # ----------------------------------------------------
        # OPENAI
        # ----------------------------------------------------
        if self.provider == "openai":

            response = self.client.chat.completions.create(

                model=self.model,

                messages=[
                    {
                        "role": "system",
                        "content": system
                    },
                    {
                        "role": "user",
                        "content": user
                    }
                ],

                temperature=temperature
            )

            return response.choices[0].message.content


        # ----------------------------------------------------
        # GEMINI
        # ----------------------------------------------------
        if self.provider == "gemini":

            prompt = (
                system
                + "\n\n"
                + user
            )

            response = self.client.models.generate_content(

                model=self.model,

                contents=prompt,

                config={
                    "temperature": temperature
                }
            )

            return response.text


    # ========================================================
    # JSON GENERATION
    # ========================================================

    def json_call(self, system, user):

        raw = self.text(
            system,
            user,
            temperature=0.0
        ).strip()

        # Remove markdown fences if model returns them
        if raw.startswith("```json"):
            raw = raw[len("```json"):].strip()

        elif raw.startswith("```"):
            raw = raw[len("```"):].strip()

        if raw.endswith("```"):
            raw = raw[:-3].strip()

        return json.loads(raw)


# ============================================================
# FINANCE SYSTEM PROMPT
# ============================================================

FINANCE_SYSTEM = """
You are a strict Personal Finance Education Assistant.

============================================================
KNOWLEDGE SOURCE POLICY
============================================================

The provided Personal Finance textbook is the ONLY knowledge
source allowed for answering user questions.

The retrieved evidence from the textbook is the ONLY factual
evidence you may use.

You MUST NOT use:
- web search
- internet sources
- external documents
- external databases
- pretrained/general world knowledge
- assumptions
- unsupported inference

The retrieved textbook evidence is your complete knowledge
boundary for this request.

============================================================
STRICT ANTI-HALLUCINATION RULES
============================================================

1. Answer ONLY using the retrieved textbook evidence.

2. NEVER add information from outside the retrieved evidence.

3. NEVER guess or complete missing information using general
   knowledge.

4. NEVER invent facts, numbers, examples, definitions, formulas,
   citations, pages, chapters, quotations, or metadata.

5. If the evidence does not contain enough information to answer
   the question, DO NOT answer from general knowledge.

6. Instead respond:

"I couldn't find sufficient evidence for this in the provided
knowledge base."

7. If only part of the question is supported, answer only that
supported part and clearly state that the remaining information
was not found in the knowledge base.

============================================================
CITATION POLICY
============================================================

Only cite sources explicitly included in the retrieved evidence.

Every citation MUST correspond to actual metadata provided by
the retrieval system.

NEVER create or guess:
- page numbers
- chapter names
- chunk IDs
- source names

============================================================
EXTERNAL SOURCE POLICY
============================================================

DO NOT browse the internet.

DO NOT call external search tools.

DO NOT use external financial information.

DO NOT use current information from outside the textbook.

If the user asks for information that is not contained in the
provided textbook evidence, refuse to answer from outside sources.

============================================================
CALCULATIONS
============================================================

For calculations:

- Use deterministic calculator results when available.
- Use only values provided by the user or retrieved evidence.
- Do not invent missing values.
- Clearly show the calculation.

If the required information is unavailable, say:

"I couldn't find sufficient evidence for this in the provided
knowledge base."

============================================================
FINAL VERIFICATION
============================================================

Before producing the answer, verify:

1. Every factual claim comes from retrieved textbook evidence.
2. Every number is supported by the evidence or calculator.
3. Every citation exists in the supplied metadata.
4. No external source was used.
5. No unsupported assumption was introduced.

If any requirement fails, remove the unsupported information or
refuse to answer.

The textbook evidence is the strict boundary of your answer.
"""


# ============================================================
# INITIALIZE LLM
# ============================================================

llm = None

if API_KEY:

    try:

        llm = LLMClient(
            LLM_PROVIDER,
            API_KEY,
            LLM_MODEL
        )

        print("=" * 70)
        print("🤖 LLM CLIENT READY")
        print("=" * 70)
        print("Provider :", LLM_PROVIDER)
        print("Model    :", LLM_MODEL)
        print("Status   : ✅ Initialized")
        print("=" * 70)

    except Exception as e:

        print("=" * 70)
        print("❌ LLM CLIENT INITIALIZATION FAILED")
        print("=" * 70)
        print(type(e).__name__, ":", e)
        print("=" * 70)

else:

    print("=" * 70)
    print("⚠️ LLM NOT CONFIGURED")
    print("=" * 70)
    print("Add the correct API key to Kaggle Secrets.")
    print("=" * 70)


# ============================================================
# INTENT HINTS
# ============================================================

HINTS = {

    "budgeting": [
        "budget",
        "spending",
        "expenses",
        "allocate income"
    ],

    "saving": [
        "save",
        "savings",
        "emergency fund"
    ],

    "debt": [
        "debt",
        "loan",
        "pay off",
        "borrowing"
    ],

    "credit": [
        "credit score",
        "credit card",
        "credit history"
    ],

    "investing": [
        "invest",
        "investment",
        "stocks",
        "bonds",
        "portfolio",
        "diversification"
    ],

    "retirement": [
        "retirement",
        "pension",
        "retire"
    ],

    "insurance": [
        "insurance",
        "coverage",
        "premium",
        "deductible"
    ],

    "taxes": [
        "tax",
        "taxes",
        "tax return"
    ],

    "financial_planning": [
        "financial plan",
        "financial planning",
        "financial goal"
    ],

    "financial_education": [
        "explain",
        "what is",
        "meaning",
        "concept"
    ],

    "book_summary": [
        "summarize",
        "chapter summary",
        "summarise"
    ],

    "comparison": [
        "compare",
        "difference between",
        "versus",
        " vs "
    ],

    "calculation": [
        "calculate",
        "percentage",
        "compound interest",
        "annual savings"
    ]
}


# ============================================================
# INTENT ROUTER
# ============================================================

class IntentRouter:

    def fallback(self, q):

        scores = {
            i: sum(
                h in q.lower()
                for h in hs
            )
            for i, hs in HINTS.items()
        }

        best = (
            max(scores, key=scores.get)
            if scores
            else "general_question"
        )

        if scores.get(best, 0) == 0:
            best = "general_question"

        return {
            "intent": best,
            "confidence": (
                0.50
                if best != "general_question"
                else 0.35
            ),
            "reason":
                "Deterministic fallback; LLM route unavailable."
        }


    def route(self, q):

        if llm is None:
            return self.fallback(q)

        prompt = """
Classify the user's question into EXACTLY ONE intent.

Available intents:
""" + ", ".join(INTENTS) + """

Return ONLY valid JSON:

{
  "intent": "...",
  "confidence": 0.0,
  "reason": "..."
}

Do not provide financial advice.
"""

        try:

            d = llm.json_call(
                prompt,
                q
            )

            intent = d.get(
                "intent",
                "general_question"
            )

            if intent not in INTENTS:
                intent = "general_question"

            return {
                "intent": intent,
                "confidence": float(
                    d.get("confidence", 0)
                ),
                "reason": str(
                    d.get("reason", "")
                )
            }

        except Exception as e:

            print(
                "⚠️ Intent Router LLM error:",
                type(e).__name__,
                ":",
                e
            )

            return self.fallback(q)


router = IntentRouter()

print("Intent Router: ✅ Ready")

In [ ]:
# ============================================================
# 🔧 PATCH: REMOVE OLD CHAPTER/SECTION DEPENDENCY
# ============================================================

import inspect

print("=" * 70)
print("🔎 CHECKING RAG ENGINE FOR CHAPTER/SECTION DEPENDENCIES")
print("=" * 70)

# Check available RAG methods
for method_name in ["search", "context", "citations"]:
    method = getattr(rag_engine, method_name, None)

    if method is None:
        print(f"⚠️ {method_name} not found")
        continue

    try:
        source = inspect.getsource(method)

        print(f"\n--- {method_name} ---")

        for i, line in enumerate(source.splitlines(), 1):
            if "chapter" in line.lower() or "section" in line.lower():
                print(f"{i:03d}: {line}")

    except Exception as e:
        print(f"⚠️ Cannot inspect {method_name}: {e}")

print("=" * 70)

## 20–21. Agents and Workflow Orchestration

The orchestrator uses a common `AgentResult` contract and can execute independent specialist agents in parallel. The final coordinator sees structured evidence, tool outputs, memory, safety instructions, and the detected intent.


In [ ]:
# ============================================================
# 20–21. AGENTS, DETERMINISTIC TOOLS, SAFETY & ORCHESTRATION
# ============================================================

@dataclass
class AgentResult:
    agent: str
    answer: str = ""
    evidence: List[Dict[str, Any]] = field(default_factory=list)
    tool_calls: List[Dict[str, Any]] = field(default_factory=list)


# ============================================================
# 🧠 MEMORY MANAGER
# ============================================================

class MemoryManager:

    def __init__(self, path):
        self.path = Path(path)
        self.short_term = []

        self.path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with sqlite3.connect(self.path) as con:
            con.execute("""
                CREATE TABLE IF NOT EXISTS memories (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    type TEXT,
                    content TEXT,
                    intent TEXT,
                    confidence REAL,
                    ts REAL
                )
            """)
            con.commit()

    def store_interaction(
        self,
        query,
        intent,
        answer,
        confidence
    ):
        self.short_term.append({
            "query": query,
            "intent": intent,
            "answer": answer,
            "ts": time.time()
        })

        self.short_term = self.short_term[-12:]

        payload = json.dumps(
            {
                "topic": intent,
                "query_summary": query[:280]
            },
            ensure_ascii=False
        )

        with sqlite3.connect(self.path) as con:
            con.execute(
                """
                INSERT INTO memories
                (type, content, intent, confidence, ts)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    "learning_context",
                    payload,
                    intent,
                    float(confidence),
                    time.time()
                )
            )
            con.commit()

    def add_preference(self, key, value):

        with sqlite3.connect(self.path) as con:
            con.execute(
                """
                INSERT INTO memories
                (type, content, intent, confidence, ts)
                VALUES (?, ?, ?, ?, ?)
                """,
                (
                    "preference",
                    json.dumps(
                        {
                            "preference": key,
                            "value": value
                        },
                        ensure_ascii=False
                    ),
                    None,
                    1.0,
                    time.time()
                )
            )
            con.commit()

    def retrieve(self, query, limit=5):

        qt = set(tokenize(query))

        with sqlite3.connect(self.path) as con:
            rows = con.execute(
                """
                SELECT id, type, content, intent, confidence, ts
                FROM memories
                ORDER BY ts DESC
                LIMIT 50
                """
            ).fetchall()

        scored = []

        for r in rows:
            overlap = len(
                qt & set(tokenize(r[2]))
            )

            scored.append((overlap, r))

        scored.sort(
            key=lambda x: (
                x[0],
                x[1][5]
            ),
            reverse=True
        )

        return [
            {
                "id": r[0],
                "memory_type": r[1],
                "content": json.loads(r[2]),
                "intent": r[3],
                "confidence": r[4]
            }
            for _, r in scored[:limit]
        ]


memory = MemoryManager(
    ARTIFACT_DIR / "memory.sqlite"
)


# ============================================================
# 🧮 DETERMINISTIC CALCULATOR
# ============================================================

class CalculatorTool:

    @staticmethod
    def monthly_to_annual(monthly):
        return float(monthly) * 12

    @staticmethod
    def percentage(value, pct):
        return float(value) * float(pct) / 100

    @staticmethod
    def compound_interest(
        principal,
        annual_rate,
        years,
        compounds_per_year=12
    ):
        r = float(annual_rate) / 100
        n = int(compounds_per_year)

        return (
            float(principal)
            * (1 + r / n) ** (n * int(years))
        )

    @staticmethod
    def debt_payoff_months(
        balance,
        annual_rate,
        payment
    ):
        b = float(balance)
        r = float(annual_rate) / 100 / 12
        p = float(payment)

        if b <= 0 or p <= 0:
            return None

        if r == 0:
            return int(math.ceil(b / p))

        if p <= b * r:
            return None

        return int(
            math.ceil(
                -math.log(1 - b * r / p)
                / math.log(1 + r)
            )
        )


# ============================================================
# 💰 ROBUST NUMBER PARSER
# ============================================================

def parse_number(value):
    """
    Safely parse financial numbers such as:

        $5,000
        $5000
        5,000
        5000
        10.5
    """

    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).strip()

    # Remove currency symbols and commas
    text = re.sub(r"[$€£,]", "", text)

    # Keep digits, decimal point and negative sign
    text = re.sub(r"[^\d.\-]", "", text)

    if not text:
        raise ValueError(
            f"Invalid numeric value: {value}"
        )

    return float(text)


# ============================================================
# 🔍 CALCULATION INFERENCE
# ============================================================

def infer_calculation(query):

    matches = re.findall(
        r"[$€£]?\s*\d+(?:,\d{3})*(?:\.\d+)?",
        query
    )

    nums = [
        parse_number(x)
        for x in matches
    ]

    q = query.lower()

    # Annual savings
    if "annual savings" in q and nums:
        return {
            "type": "annual_savings",
            "monthly": nums[-1]
        }

    # Compound interest
    if "compound interest" in q and len(nums) >= 3:
        return {
            "type": "compound_interest",
            "principal": nums[0],
            "annual_rate": nums[1],
            "years": int(nums[2])
        }

    # Percentage
    if "%" in query and len(nums) >= 2:
        return {
            "type": "percentage",
            "value": nums[0],
            "percent": nums[1]
        }

    return {}


# ============================================================
# 🛡️ SAFETY
# ============================================================

def safety_notes(query):

    notes = [
        "This is financial education, not personalized financial advice.",
        "Investment explanations do not recommend specific securities."
    ]

    if any(
        x in query.lower()
        for x in [
            "tax",
            "law",
            "regulation",
            "rate",
            "mortgage"
        ]
    ):
        notes.append(
            "Current laws, rates, and regulations may require external verification."
        )

    return notes


# ============================================================
# 📚 RESEARCH AGENT
# ============================================================

class ResearchAgent:

    name = "research_agent"

    def run(self, query, state):

        pack = rag_engine.search(query)

        return AgentResult(
            self.name,
            rag_engine.context(pack["reranked"]),
            pack["reranked"]
        )


# ============================================================
# 🧮 CALCULATOR AGENT
# ============================================================

class CalculatorAgent:

    name = "calculator_agent"

    def run(self, query, state):

        c = state.get(
            "calculation",
            {}
        )

        if c.get("type") == "annual_savings":

            out = CalculatorTool.monthly_to_annual(
                c["monthly"]
            )

            return AgentResult(
                self.name,
                f"Annual savings = ${out:,.2f}.",
                tool_calls=[
                    {
                        "tool": "monthly_to_annual",
                        "input": c,
                        "output": out
                    }
                ]
            )

        if c.get("type") == "compound_interest":

            out = CalculatorTool.compound_interest(
                c["principal"],
                c["annual_rate"],
                c["years"]
            )

            return AgentResult(
                self.name,
                f"Future value = ${out:,.2f}.",
                tool_calls=[
                    {
                        "tool": "compound_interest",
                        "input": c,
                        "output": out
                    }
                ]
            )

        if c.get("type") == "percentage":

            out = CalculatorTool.percentage(
                c["value"],
                c["percent"]
            )

            return AgentResult(
                self.name,
                f"Result = ${out:,.2f}.",
                tool_calls=[
                    {
                        "tool": "percentage",
                        "input": c,
                        "output": out
                    }
                ]
            )

        return AgentResult(
            self.name,
            "No deterministic calculation was detected."
        )


# ============================================================
# 🤖 SPECIALIZED AGENTS
# ============================================================

class FinancialEducationAgent(ResearchAgent):
    name = "financial_education_agent"


class BudgetAgent(ResearchAgent):
    name = "budget_agent"


class DebtAgent(ResearchAgent):
    name = "debt_agent"


class InvestmentAgent(ResearchAgent):
    name = "investment_education_agent"


class SummaryAgent(ResearchAgent):
    name = "summary_agent"


# ============================================================
# 🧭 AGENT ROUTING
# ============================================================

AGENT_ROUTING = {

    "budgeting": [
        "budget_agent",
        "research_agent"
    ],

    "saving": [
        "financial_education_agent",
        "research_agent"
    ],

    "debt": [
        "debt_agent",
        "research_agent"
    ],

    "credit": [
        "financial_education_agent",
        "research_agent"
    ],

    "investing": [
        "investment_education_agent",
        "research_agent"
    ],

    "retirement": [
        "financial_education_agent",
        "research_agent"
    ],

    "insurance": [
        "financial_education_agent",
        "research_agent"
    ],

    "taxes": [
        "financial_education_agent",
        "research_agent"
    ],

    "financial_planning": [
        "financial_education_agent",
        "research_agent"
    ],

    "financial_education": [
        "financial_education_agent",
        "research_agent"
    ],

    "book_summary": [
        "summary_agent",
        "research_agent"
    ],

    "comparison": [
        "financial_education_agent",
        "research_agent"
    ],

    "calculation": [
        "calculator_agent",
        "research_agent"
    ],

    "general_question": [
        "research_agent"
    ],

    "unsupported": [
        "research_agent"
    ]
}


# ============================================================
# 🎼 ORCHESTRATOR
# ============================================================

class Orchestrator:

    def __init__(self):

        self.agents = {

            "research_agent":
                ResearchAgent(),

            "financial_education_agent":
                FinancialEducationAgent(),

            "budget_agent":
                BudgetAgent(),

            "debt_agent":
                DebtAgent(),

            "investment_education_agent":
                InvestmentAgent(),

            "summary_agent":
                SummaryAgent(),

            "calculator_agent":
                CalculatorAgent()
        }


    # --------------------------------------------------------
    # FINAL SYNTHESIS
    # --------------------------------------------------------

    def synthesize(
        self,
        query,
        route,
        memory_items,
        results,
        sources,
        tools,
        grounded
    ):

        if llm is None:

            return (
                "LLM is not configured. "
                "Retrieved evidence and execution metadata "
                "are available, but no generated answer is produced."
            )

        # IMPORTANT:
        # chapter/section were intentionally removed
        # from chunk metadata.

        evidence = [
            (
                f"[{x.get('source', 'Personal Finance')} "
                f"| Page: {x.get('page', 'N/A')} "
                f"| Chunk: {x.get('chunk_id', 'N/A')}]\n"
                f"{x.get('text', '')}"
            )
            for x in sources
        ]

        payload = {

            "query": query,

            "intent": route,

            "memory": memory_items,

            "agents": {
                r.agent: r.answer
                for r in results
            },

            "tools": tools,

            "grounded": grounded,

            "evidence": evidence,

            "safety": safety_notes(query)
        }

        return llm.text(
            FINANCE_SYSTEM
            + "\nAnswer from this structured context:\n"
            + json.dumps(
                payload,
                ensure_ascii=False
            ),
            "",
            temperature=0.15
        )


    # --------------------------------------------------------
    # HANDLE QUERY
    # --------------------------------------------------------

    def handle(self, query):

        t0 = time.perf_counter()

        # Intent routing
        route = router.route(query)

        # Memory
        mem = memory.retrieve(query)

        # Calculation detection
        calc = infer_calculation(query)

        # Select agents
        selected = list(
            AGENT_ROUTING.get(
                route["intent"],
                ["research_agent"]
            )
        )

        # Add calculator when needed
        if calc:
            selected = list(
                dict.fromkeys(
                    selected + ["calculator_agent"]
                )
            )

        state = {
            "calculation": calc
        }

        results = []

        # Parallel execution
        with ThreadPoolExecutor(
            max_workers=min(
                4,
                len(selected)
            )
        ) as ex:

            jobs = {
                ex.submit(
                    self.agents[a].run,
                    query,
                    state
                ): a
                for a in selected
            }

            for f in as_completed(jobs):

                results.append(
                    f.result()
                )

        # ----------------------------------------------------
        # Collect evidence and tools
        # ----------------------------------------------------

        sources = []
        seen = set()
        tools = []

        for r in results:

            tools.extend(
                r.tool_calls
            )

            for e in r.evidence:

                chunk_id = e.get(
                    "chunk_id"
                )

                if chunk_id not in seen:

                    seen.add(chunk_id)

                    sources.append(e)

        # Retrieval statistics
        retrieval_pack = (
            rag_engine.search(query)
            if retriever is not None
            else {
                "candidates": [],
                "reranked": []
            }
        )

        grounded = bool(
            sources
        )

        # Final answer
        answer = self.synthesize(
            query,
            route,
            mem,
            results,
            sources,
            tools,
            grounded
        )

        # Store interaction
        memory.store_interaction(
            query,
            route["intent"],
            answer,
            route["confidence"]
        )

        latency = (
            time.perf_counter()
            - t0
        )

        # Trace
        trace = {

            "query": query,

            "intent": route["intent"],

            "intent_confidence":
                route["confidence"],

            "selected_agents":
                selected,

            "retrieved_chunks":
                len(
                    retrieval_pack["candidates"]
                ),

            "reranked_chunks":
                len(
                    retrieval_pack["reranked"]
                ),

            "memory_used":
                bool(mem),

            "tools_used":
                [
                    t["tool"]
                    for t in tools
                ],

            "latency":
                latency,

            "sources":
                rag_engine.citations(
                    sources[:TOP_K_RERANK]
                ),

            "grounded":
                grounded
        }

        return {

            "answer": answer,

            "intent": route,

            "agents": selected,

            "memory": mem,

            "tools": tools,

            "sources":
                rag_engine.citations(
                    sources[:TOP_K_RERANK]
                ),

            "trace": trace
        }


# ============================================================
# 🚀 CREATE ORCHESTRATOR
# ============================================================

orchestrator = Orchestrator()


# ============================================================
# 💬 PUBLIC FINANCE QUERY FUNCTION
# ============================================================

def ask_finance(
    query,
    verbose=True
):

    if not query.strip():
        raise ValueError(
            "Query cannot be empty."
        )

    res = orchestrator.handle(
        query
    )

    if verbose:

        print(
            "Intent:",
            res["intent"]
        )

        print(
            "Agents:",
            res["agents"]
        )

        print(
            "Memory used:",
            bool(res["memory"])
        )

        print(
            "Tools:",
            [
                x["tool"]
                for x in res["tools"]
            ]
        )

        print(
            "Sources:",
            len(res["sources"])
        )

        print(
            "Latency:",
            round(
                res["trace"]["latency"],
                3
            ),
            "s"
        )

        print(
            "\nANSWER\n",
            res["answer"]
        )

        if res["sources"]:

            print(
                "\nSOURCES"
            )

            for s in res["sources"]:

                print(
                    f"- "
                    f"{s.get('source', 'Personal Finance')} "
                    f"| Page {s.get('page', 'N/A')} "
                    f"| {s.get('chunk_id', 'N/A')}"
                )

    return res


# ============================================================
# ✅ VALIDATION
# ============================================================

print("=" * 70)
print("🤖 AGENTS & ORCHESTRATOR READY")
print("=" * 70)

print(
    "Agents:",
    list(orchestrator.agents.keys())
)

print(
    "Routing intents:",
    len(AGENT_ROUTING)
)

print(
    "Metadata schema:",
    ["chunk_id", "source", "page", "text"]
)

print(
    "Chapter dependency:",
    "❌ Removed"
)

print(
    "Section dependency:",
    "❌ Removed"
)

print(
    "Calculator currency parsing:",
    "✅ Enabled"
)

print("=" * 70)

## 22–24. Memory Demonstration, Tools, and Safety

Long-term memory is persisted in SQLite under the Kaggle working directory. The demonstration uses synthetic learning context. The calculator is deterministic Python code and is included in the agent graph.


In [ ]:
# Synthetic memory demonstration and deterministic tool checks
memory.add_preference("preferred_explanation_style", "simple")
memory.add_preference("topics_studied", ["budgeting"])

tool_checks = pd.DataFrame([{
    "annual_$1200": CalculatorTool.monthly_to_annual(1200),
    "10pct_of_$5000": CalculatorTool.percentage(5000,10),
    "compound_$1000_5pct_10y": CalculatorTool.compound_interest(1000,5,10),
    "debt_payoff_months": CalculatorTool.debt_payoff_months(5000,12,250)
}])
display(tool_checks)
print("Memory recall:", memory.retrieve("What should I study after budgeting?", limit=5))


In [ ]:
# ============================================================
# 💾 GOOGLE COLAB — AUTO SAVE / RESUME SYSTEM
# ============================================================

from pathlib import Path
import json
import os
import time

# Persistent storage
PROJECT_DIR = Path("/content/personal-finance-agentic-rag-assistant")

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
OUTPUT_DIR = PROJECT_DIR / "outputs"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

STATE_FILE = CHECKPOINT_DIR / "pipeline_state.json"


# ============================================================
# STATE MANAGEMENT
# ============================================================

def save_state(stage, **extra):

    state = {
        "stage": stage,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        **extra
    }

    STATE_FILE.write_text(
        json.dumps(state, indent=2),
        encoding="utf-8"
    )

    print(f"💾 Checkpoint saved: {stage}")


def load_state():

    if not STATE_FILE.exists():
        return {
            "stage": "START"
        }

    try:
        return json.loads(
            STATE_FILE.read_text(encoding="utf-8")
        )

    except Exception:
        return {
            "stage": "START"
        }


state = load_state()

print("=" * 70)
print("💾 PERSONAL FINANCE AGENTIC RAG — AUTO RESUME")
print("=" * 70)

print("Last checkpoint :", state.get("stage"))
print("Saved at        :", state.get("timestamp", "Never"))

print("=" * 70)

## 25–26. End-to-End Demonstrations

Run realistic queries only after the PDF and API configuration are available. The examples expose intent, agents, memory, sources, tools, and latency.


In [ ]:
# ============================================================
# 25–26. END-TO-END DEMONSTRATION
# ============================================================

DEMO_QUERIES = [
    "What is a budget and why is it important?",
    "Explain compound interest in simple terms.",
    "How should someone think about managing debt?",
    "I earn $5,000 and want to save $1,000 per month. Calculate my annual savings.",
    "Based on what I previously told you, what financial topic should I study next?",
    "Create a personal finance learning plan.",
    "Ask the book something it cannot answer: what is the population of Mars?"
]

demo_records = []

print("=" * 100)
print("🚀 END-TO-END DEMONSTRATION")
print("=" * 100)

if not llm:
    print("❌ LLM client is not available.")
elif not retriever:
    print("❌ Retriever is not available.")
else:

    for i, q in enumerate(DEMO_QUERIES, 1):

        print("\n" + "=" * 100)
        print(f"QUERY {i}/{len(DEMO_QUERIES)}")
        print("=" * 100)
        print("🧑 User:", q)

        try:
            r = ask_finance(q, verbose=True)

            record = {
                "query": q,
                "intent": r.get("intent", {}).get("intent", "unknown"),
                "agents": r.get("agents", []),
                "memory_used": bool(r.get("memory")),
                "tools": [
                    x.get("tool")
                    for x in r.get("tools", [])
                    if isinstance(x, dict)
                ],
                "sources": len(r.get("sources", [])),
                "latency": r.get("trace", {}).get("latency", 0),
                "answer": r.get("answer", "")
            }

            demo_records.append(record)

            print("\n✅ Query completed")

        except Exception as e:

            print(f"\n❌ Query failed: {type(e).__name__}: {e}")

            demo_records.append({
                "query": q,
                "intent": "ERROR",
                "agents": [],
                "memory_used": False,
                "tools": [],
                "sources": 0,
                "latency": 0,
                "answer": f"ERROR: {e}"
            })


# ============================================================
# 📊 DEMO RESULTS
# ============================================================

if demo_records:

    demo_df = pd.DataFrame(demo_records)

    print("\n" + "=" * 100)
    print("📊 END-TO-END DEMO SUMMARY")
    print("=" * 100)

    print(f"Total queries : {len(demo_df)}")
    print(
        f"Successful    : "
        f"{(demo_df['intent'] != 'ERROR').sum()}"
    )
    print(
        f"Failed        : "
        f"{(demo_df['intent'] == 'ERROR').sum()}"
    )

    print("\n🎯 Intent Distribution:")
    display(
        demo_df["intent"]
        .value_counts()
        .to_frame("count")
    )

    print("\n📋 Demo Records:")
    display(
        demo_df[
            [
                "query",
                "intent",
                "agents",
                "memory_used",
                "tools",
                "sources",
                "latency"
            ]
        ]
    )

    print("\n" + "=" * 100)
    print("✅ END-TO-END DEMONSTRATION FINISHED")
    print("=" * 100)

🚀 END-TO-END DEMONSTRATION
❌ LLM client is not available.


## 27. Evaluation Dataset

The dataset contains 40 labeled questions across concepts, how-to questions, comparisons, calculations, summaries, planning, and unsupported prompts.


In [ ]:
EVAL_DATA = [
("What is a budget and why is it important?","budgeting","budget","concept","easy"),
("How can someone track monthly expenses?","budgeting","expenses","how_to","easy"),
("Explain the purpose of an emergency fund.","saving","emergency fund","concept","easy"),
("What is compound interest?","financial_education","compound interest","concept","easy"),
("How can someone manage debt responsibly?","debt","debt","how_to","medium"),
("What factors can affect the cost of borrowing?","debt","borrowing","concept","medium"),
("What is a credit score?","credit","credit score","concept","easy"),
("Why might a credit history matter?","credit","credit history","concept","easy"),
("What is diversification?","investing","diversification","concept","easy"),
("Explain the difference between stocks and bonds.","comparison","stocks bonds","comparison","medium"),
("What is asset allocation?","investing","asset allocation","concept","medium"),
("Explain retirement planning.","retirement","retirement","concept","easy"),
("Why can retirement goals affect saving decisions?","retirement","retirement saving","reasoning","medium"),
("What is insurance for?","insurance","insurance","concept","easy"),
("Explain the role of an insurance deductible.","insurance","deductible","concept","medium"),
("Why should tax information be checked for the current year?","taxes","taxes","safety","medium"),
("Create a general financial learning plan.","financial_planning","financial planning","planning","medium"),
("Compare saving and investing.","comparison","saving investing","comparison","medium"),
("Calculate annual savings for $1,000 saved every month.","calculation","annual savings","calculation","easy"),
("What is a good way to think about financial goals?","financial_planning","goals","planning","easy"),
("Summarize the chapter about insurance.","book_summary","insurance","summary","medium"),
("Summarize the chapter about investing.","book_summary","investing","summary","medium"),
("Explain opportunity cost in personal finance.","financial_education","opportunity cost","concept","medium"),
("How can someone prioritize financial goals?","financial_planning","financial goals","how_to","medium"),
("What is net worth?","financial_education","net worth","concept","easy"),
("What is the difference between needs and wants?","comparison","needs wants","comparison","easy"),
("How can a person prepare for unexpected expenses?","saving","unexpected expenses","how_to","medium"),
("Why might high-interest debt deserve attention?","debt","high interest debt","reasoning","medium"),
("How does compound growth work over time?","financial_education","compound growth","concept","medium"),
("What should a beginner learn before investing?","investing","investment education","education","medium"),
("What is the purpose of a financial plan?","financial_planning","financial plan","concept","easy"),
("What is the difference between a fixed and variable expense?","budgeting","expenses","comparison","easy"),
("How can someone approach paying off multiple debts?","debt","debt payoff","how_to","medium"),
("What makes a financial goal measurable?","financial_planning","financial goals","concept","medium"),
("According to the book, what is the population of Mars?","unsupported","mars","unsupported","easy"),
("According to the book, who won the 2026 World Cup?","unsupported","world cup","unsupported","easy"),
("Calculate 10% of $5,000.","calculation","percentage","calculation","easy"),
("Calculate the future value of $1,000 at 5% for 10 years with monthly compounding.","calculation","compound interest","calculation","medium"),
("What financial topic should I study next after budgeting?","financial_planning","learning path","personalized_learning","medium"),
("Explain why diversification can reduce concentration risk.","investing","diversification risk","reasoning","hard")
]
eval_df=pd.DataFrame(EVAL_DATA,columns=["question","expected_intent","expected_topic","question_type","difficulty"])
print("Evaluation questions:",len(eval_df))
display(eval_df.head())


## 28–31. Retrieval, Intent, Agent Routing, Memory Evaluation

The metrics below are computed from real execution. Retrieval uses a transparent topic-match proxy, intent uses precision/recall/F1, and unsupported prompts are separately inspected.


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

def retrieval_eval(frame, k=TOP_K_RERANK):
    rows=[]
    if retriever is None: return pd.DataFrame(), {"Hit@K":np.nan,"MRR":np.nan,"evaluated":0}
    for _,r in frame.iterrows():
        pack=rag_engine.search(r["question"]); got=pack["reranked"][:k]
        expected=set(tokenize(r["expected_topic"])); hit_rank=None
        for rank,g in enumerate(got,1):
            tokens=set(tokenize(g["text"]))
            if expected and expected.issubset(tokens):
                hit_rank=rank; break
        rows.append({"question":r["question"],"expected_topic":r["expected_topic"],
                     "hit@k":int(hit_rank is not None),
                     "reciprocal_rank":1/hit_rank if hit_rank else 0,
                     "top_score":got[0]["fused_score"] if got else 0})
    d=pd.DataFrame(rows)
    return d, {"Hit@K":float(d["hit@k"].mean()),"MRR":float(d["reciprocal_rank"].mean()),"evaluated":len(d)}

retrieval_detail,retrieval_summary=retrieval_eval(eval_df)
display(retrieval_detail.head(10)); print(retrieval_summary)
retrieval_detail.to_csv(OUTPUT_DIR/"retrieval_evaluation.csv",index=False)

def intent_eval(frame):
    rows=[]
    for _,r in frame.iterrows():
        p=router.route(r["question"])
        rows.append({"question":r["question"],"expected_intent":r["expected_intent"],
                     "predicted_intent":p["intent"],"confidence":p["confidence"],
                     "correct":p["intent"]==r["expected_intent"]})
    d=pd.DataFrame(rows)
    labs=sorted(set(d.expected_intent)|set(d.predicted_intent))
    p,rc,f1,_=precision_recall_fscore_support(d.expected_intent,d.predicted_intent,labels=labs,average="macro",zero_division=0)
    summary={"accuracy":float(accuracy_score(d.expected_intent,d.predicted_intent)),
             "precision_macro":float(p),"recall_macro":float(rc),"f1_macro":float(f1),"evaluated":len(d)}
    return d,summary

intent_detail,intent_summary=intent_eval(eval_df)
display(intent_detail.head(10)); print(intent_summary)
intent_detail.to_csv(OUTPUT_DIR/"intent_evaluation.csv",index=False)

routing_rows=[]
for _, r in eval_df.iterrows():
    predicted_intent = router.route(r["question"])["intent"]
    expected_agents = set(AGENT_ROUTING.get(r["expected_intent"], ["research_agent"]))
    predicted_agents = set(AGENT_ROUTING.get(predicted_intent, ["research_agent"]))
    routing_rows.append({
        "question": r["question"],
        "expected_intent": r["expected_intent"],
        "predicted_intent": predicted_intent,
        "expected_agents": sorted(expected_agents),
        "predicted_agents": sorted(predicted_agents),
        "routing_correct": expected_agents == predicted_agents
    })
routing_detail=pd.DataFrame(routing_rows)
routing_summary={"routing_accuracy":float(routing_detail.routing_correct.mean()),"evaluated":len(routing_detail)}
routing_detail.to_csv(OUTPUT_DIR/"agent_evaluation.csv",index=False)
print(routing_summary)

memory_test=memory.retrieve("budgeting study topic",limit=5)
memory_summary={"retrieved_items":len(memory_test),"contains_budgeting_context":any("budget" in json.dumps(x).lower() for x in memory_test)}
print("Memory evaluation:",memory_summary)


## 32. Baseline Comparison: LLM-only vs RAG vs Agentic RAG

The comparison uses a small demonstration set to limit API cost. The notebook records actual latency and whether source citations were available. It does not fabricate answer-quality scores.


In [ ]:

BASELINE_QUERIES = [
    "What is a budget and why is it important?",
    "Explain compound interest in simple terms."


]

def llm_only(q):
    return llm.text(FINANCE_SYSTEM,q,temperature=0.15)

def rag_only(q):
    pack=rag_engine.search(q)
    context=rag_engine.context(pack["reranked"])
    return llm.text(FINANCE_SYSTEM, f"QUESTION:\n{q}\n\nEVIDENCE:\n{context}\n\nAnswer only from the evidence and cite its metadata.", temperature=0.15)

baseline_rows=[]
if llm and retriever:
    for q in BASELINE_QUERIES:
        t=time.perf_counter(); a0=llm_only(q); t0=time.perf_counter()-t
        t=time.perf_counter(); a1=rag_only(q); t1=time.perf_counter()-t
        t=time.perf_counter(); a2=ask_finance(q,False); t2=time.perf_counter()-t
        baseline_rows.append({
            "query":q,"llm_only_latency":t0,"rag_latency":t1,"agentic_rag_latency":t2,
            "rag_sources":int(bool(rag_engine.search(q)["reranked"])),
            "agentic_sources":int(bool(a2["sources"])),
            "agentic_tools":len(a2["tools"])
        })
baseline_df=pd.DataFrame(baseline_rows)
display(baseline_df)
baseline_df.to_csv(OUTPUT_DIR/"baseline_comparison.csv",index=False)


##

33. Hallucination / Unsupported-Question Testing

The retrieval gate and response prompt are designed not to force answers for unsupported questions. The cell records the actual response and whether a refusal signal was observed.


In [ ]:
unsupported=[
    "According to the book, what is the population of Mars?",
    "According to the book, who won the 2026 World Cup?"
]
hall_rows=[]
for q in unsupported:
    if llm and retriever:
        t=time.perf_counter(); r=ask_finance(q,False); lat=time.perf_counter()-t
        text=r["answer"].lower()
        refusal=any(s in text for s in ["couldn't find sufficient evidence","insufficient evidence","not in the provided knowledge base","cannot find"])
        hall_rows.append({"query":q,"grounded":bool(r["sources"]),"refusal_signal":int(refusal),"latency":lat,"answer":r["answer"]})
hallucination_df=pd.DataFrame(hall_rows)
display(hallucination_df[["query","grounded","refusal_signal","latency"]] if len(hallucination_df) else hallucination_df)
hallucination_df.to_csv(OUTPUT_DIR/"hallucination_test.csv",index=False)


## 34–35. Observability, Reports, and Dashboard

Execution traces are stored as structured JSON. Automated reports are written to `/kaggle/working/outputs`.


In [ ]:
class ReportGenerator:
    def __init__(self, outdir):
        self.outdir=Path(outdir); self.outdir.mkdir(parents=True,exist_ok=True)

    def markdown(self, query, result, filename="financial_report.md"):
        lines=[
            "# Personal Finance Education Report","",
            "## Query",query,"",
            f"## Intent\n{result['intent']['intent']} (confidence={result['intent']['confidence']:.2f})","",
            "## Answer",result["answer"],"",
            "## Agents",", ".join(result["agents"]),"",
            "## Tools",", ".join(x["tool"] for x in result["tools"]) if result["tools"] else "None","",
            "## Sources"
        ]
        lines += [f"- {s['source']}   — Page {s['page']} — {s['chunk_id']}" for s in result["sources"]] or ["- No grounded sources."]
        p=self.outdir/filename; p.write_text("\n".join(lines),encoding="utf-8"); return p

    def json(self,obj,filename):
        p=self.outdir/filename; p.write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding="utf-8"); return p

reporter=ReportGenerator(OUTPUT_DIR)

trace_df=pd.DataFrame([r["trace"] for r in [orchestrator.handle(q) for q in BASELINE_QUERIES]]) if (llm and retriever) else pd.DataFrame()
if len(trace_df):
    trace_df.to_json(OUTPUT_DIR/"execution_traces.json",orient="records",indent=2)
    print("Trace rows:",len(trace_df))

performance_summary=pd.DataFrame([
    ["Pages",len(pages)],["Chunks",len(chunks)],
    ["Intent Accuracy",intent_summary.get("accuracy",np.nan)],
    ["Intent Precision Macro",intent_summary.get("precision_macro",np.nan)],
    ["Intent Recall Macro",intent_summary.get("recall_macro",np.nan)],
    ["Intent F1 Macro",intent_summary.get("f1_macro",np.nan)],
    ["Retrieval Hit@K",retrieval_summary.get("Hit@K",np.nan)],
    ["Retrieval MRR",retrieval_summary.get("MRR",np.nan)],
    ["Agent Routing Accuracy",routing_summary["routing_accuracy"]],
    ["Average Agentic Latency",baseline_df["agentic_rag_latency"].mean() if len(baseline_df) else np.nan]
],columns=["metric","value"])
display(performance_summary)
performance_summary.to_csv(OUTPUT_DIR/"performance_summary.csv",index=False)


In [ ]:
# 35. Professional visualizations generated only from measured data
plt.figure(figsize=(8,4))
plt.bar(["Pages","Chunks"],[len(pages),len(chunks)])
plt.title("Knowledge Base Size"); plt.ylabel("Count"); plt.show()

if len(retrieval_detail):
    plt.figure(figsize=(8,4))
    plt.hist(retrieval_detail["top_score"],bins=10)
    plt.title("Top Fused Retrieval Score Distribution"); plt.xlabel("Fused score"); plt.ylabel("Questions"); plt.show()

if len(intent_detail):
    labs=sorted(set(intent_detail.expected_intent)|set(intent_detail.predicted_intent))
    cm=confusion_matrix(intent_detail.expected_intent,intent_detail.predicted_intent,labels=labs)
    fig,ax=plt.subplots(figsize=(11,8))
    ConfusionMatrixDisplay(cm,display_labels=labs).plot(ax=ax,xticks_rotation=45,colorbar=False)
    ax.set_title("Intent Classification Confusion Matrix"); plt.tight_layout(); plt.show()

if len(baseline_df):
    means=baseline_df[["llm_only_latency","rag_latency","agentic_rag_latency"]].mean()
    plt.figure(figsize=(8,4))
    plt.bar(["LLM-only","RAG","Agentic RAG"],means.values)
    plt.title("Measured Average Latency"); plt.ylabel("Seconds"); plt.show()

if llm and retriever:
    demo=ask_finance("I earn $5,000 per month and want to create a budget while paying off debt.",False)
    reporter.markdown("I earn $5,000 per month and want to create a budget while paying off debt.",demo)
    reporter.json(demo,"financial_summary.json")

print("Outputs:", [p.name for p in sorted(OUTPUT_DIR.iterdir())])


## 36. Optional Gradio Interface

The UI is optional. The notebook remains fully usable through `ask_finance()` if Gradio is unavailable or Kaggle networking causes a compatibility issue.


## 37. Final Results

The following summary is generated from the actual runtime state. Missing setup is shown explicitly rather than replaced by invented results.


In [ ]:
# ============================================================
# 🤖 PERSONAL FINANCE CHATBOT
# Agentic RAG + PDF Grounded Sources
# Colab / Gradio Compatible
# ============================================================

import gradio as gr


# ============================================================
# 1️⃣ CHAT FUNCTION
# ============================================================

def chat_fn(message, history):

    message = (message or "").strip()

    # Empty message
    if not message:
        return "", history or []

    history = history or []

    try:

        # ----------------------------------------------------
        # Check system
        # ----------------------------------------------------

        if not llm or not retriever:

            answer = (
                "⚠️ The Personal Finance Assistant is not configured.\n\n"
                "Please make sure the LLM and PDF retriever are available."
            )

        else:

            # ------------------------------------------------
            # Agentic RAG
            # ------------------------------------------------

            result = ask_finance(
                message,
                False
            )

            answer = result["answer"]

            # ------------------------------------------------
            # PDF SOURCES ONLY
            # ------------------------------------------------

            sources = result.get("sources", [])

            if sources:

                answer += "\n\n### 📚 Sources\n"

                for s in sources:

                    answer += (
                        f"- {s.get('source', 'Personal Finance.pdf')} "
                        f"| Page {s.get('page', 'N/A')} "
                        f"| {s.get('chunk_id', 'N/A')}\n"
                    )

            else:

                answer += (
                    "\n\n### 📚 Sources\n"
                    "- No grounded source was retrieved from the PDF."
                )

        # ----------------------------------------------------
        # Add USER message
        # ----------------------------------------------------

        history.append({
            "role": "user",
            "content": message
        })

        # ----------------------------------------------------
        # Add ASSISTANT message
        # ----------------------------------------------------

        history.append({
            "role": "assistant",
            "content": answer
        })

        return "", history


    except Exception as e:

        error_answer = (
            "⚠️ I couldn't process your question.\n\n"
            f"Error: `{type(e).__name__}`\n\n"
            f"`{str(e)}`"
        )

        history.append({
            "role": "user",
            "content": message
        })

        history.append({
            "role": "assistant",
            "content": error_answer
        })

        return "", history


# ============================================================
# 2️⃣ CLEAR CHAT
# ============================================================

def clear_chat():
    return []


# ============================================================
# 3️⃣ BUILD CHATBOT UI
# ============================================================

with gr.Blocks(
    title="AI Personal Finance Assistant"
) as demo:

    # --------------------------------------------------------
    # Header
    # --------------------------------------------------------

    gr.Markdown(
        """
# 💰 AI Personal Finance Assistant

### Agentic RAG Personal Finance Chatbot

Ask questions about **personal finance** using the
provided **Personal Finance PDF knowledge base**.

🔒 **Knowledge Base:** Personal Finance.pdf
🧠 **Architecture:** Agentic RAG
📚 **Sources:** PDF only
"""
    )


    # --------------------------------------------------------
    # Chatbot
    # --------------------------------------------------------

    chatbot = gr.Chatbot(
        label="Finance Assistant",
        height=550
    )


    # --------------------------------------------------------
    # Input Area
    # --------------------------------------------------------

    with gr.Row():

        message = gr.Textbox(
            label="Your Question",
            placeholder="Ask a personal finance question...",
            lines=2,
            scale=8
        )

        send = gr.Button(
            "Send 🚀",
            scale=1
        )


    # --------------------------------------------------------
    # Clear Button
    # --------------------------------------------------------

    clear = gr.Button(
        "🗑️ Clear Chat"
    )


    # ========================================================
    # 4️⃣ SEND BUTTON
    # ========================================================

    send.click(
        fn=chat_fn,
        inputs=[
            message,
            chatbot
        ],
        outputs=[
            message,
            chatbot
        ]
    )


    # ========================================================
    # 5️⃣ ENTER KEY
    # ========================================================

    message.submit(
        fn=chat_fn,
        inputs=[
            message,
            chatbot
        ],
        outputs=[
            message,
            chatbot
        ]
    )


    # ========================================================
    # 6️⃣ CLEAR CHAT
    # ========================================================

    clear.click(
        fn=clear_chat,
        inputs=None,
        outputs=chatbot
    )


# ============================================================
# 7️⃣ LAUNCH
# ============================================================

print("=" * 70)
print("🤖 PERSONAL FINANCE CHATBOT")
print("=" * 70)

print("✅ Gradio interface created")
print("✅ Agentic RAG connected")
print("✅ PDF sources enabled")
print("🔒 External web sources are NOT used")
print("🚀 Launching chatbot...")
print("=" * 70)

demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False,
    debug=True
)


In [ ]:
print("="*90)
print("PROJECT FINAL RESULTS")
print("="*90)
print("Problem solved        : Educational Personal Finance Assistant using a true multi-agent workflow")
print("Data source           :", str(PDF_FILE) if PDF_FILE else "PDF not configured")
print("Pages                 :", len(pages))
print("Chunks                :", len(chunks))
print("Embedding model       :", EMBEDDING_MODEL)
print("Vector store          : FAISS IndexFlatIP")
print("Hybrid retrieval      : Semantic + BM25")
print("Reranker              :", RERANKER_MODEL if reranker_enabled else "Unavailable — fallback to fused ranking")
print("LLM                   :", f"{LLM_PROVIDER}/{LLM_MODEL}" if llm else "Not configured")
print("Memory                : Short-term session + persistent SQLite")
print("Tools                 : Deterministic calculator")
print("Orchestration         : Parallel specialists + coordinator synthesis")
print("Automated actions     : Markdown / JSON / CSV + dashboard")
print("Evaluation questions  :", len(eval_df))
print()
display(performance_summary)


## 38–40. Limitations, Future Work, Conclusion

### Limitations
- PDF extraction and heading detection depend on source formatting.
- The textbook may not reflect current laws, tax rules, rates, or regulations.
- The LLM depends on an externally configured provider.
- Topic-based retrieval Hit@K/MRR is an explicit proxy, not a human gold-label benchmark.
- LLM-as-judge answer quality is not counted unless it is actually executed.

### Future improvements
- Curated human-labeled retrieval benchmark.
- Stronger domain reranker.
- Formal faithfulness/citation evaluation with human spot checks.
- Current external-data tools with date/source tracking.
- Richer semantic long-term memory.
- Production API, authentication, privacy, monitoring, and deployment.

### Conclusion

The notebook demonstrates a complete Agentic RAG architecture: **RAG + Intent Routing + Multi-Agent System + Memory + LLM + Tools + Workflow Orchestration + Automated Actions + Evaluation + Kaggle GPU support**, while keeping API secrets out of source code and avoiding fabricated results.


In [ ]:
# Final reproducibility check
print("Reproducibility checklist")
print("Python version      :", sys.version.split()[0])
print("PyTorch version     :", getattr(torch,"__version__","N/A"))
print("CUDA available      :", DEVICE=="cuda")
print("GPU                 :", GPU_NAME)
print("Embedding model     :", EMBEDDING_MODEL)
print("Vector DB            : FAISS")
print("LLM provider        :", LLM_PROVIDER)
print("Documents/pages     :", len(pages))
print("Chunks              :", len(chunks))
print("Evaluation questions:", len(eval_df))
print("Output directory    :", OUTPUT_DIR)
print("\nReady for top-to-bottom Kaggle execution after PDF and API Secret configuration.")


In [ ]:
!pip -q install pyngrok

In [ ]:
!pip install -q "gradio>=5,<6"

In [ ]:
from getpass import getpass
from pyngrok import ngrok
import threading
import time

# ============================================================
# 🔐 NGROK AUTH
# ============================================================

NGROK_TOKEN = getpass("Enter your ngrok authtoken: ")

ngrok.set_auth_token(NGROK_TOKEN)

# ============================================================
# 🌐 START GRADIO
# ============================================================

def launch_gradio():
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        debug=True
    )

thread = threading.Thread(
    target=launch_gradio,
    daemon=True
)

thread.start()

# Give Gradio time to start
time.sleep(5)

# ============================================================
# 🌍 START NGROK
# ============================================================

public_url = ngrok.connect(
    7860,
    "http"
)

print("=" * 70)
print("🚀 PERSONAL FINANCE AGENTIC RAG — DEPLOYED")
print("=" * 70)

print("Local URL  : http://127.0.0.1:7860")
print("Public URL :", public_url)

print("=" * 70)
print("✅ Open the Public URL in your browser.")
print("🔒 Knowledge Base: Personal Finance.pdf")
print("📚 Sources: PDF only")
print("=" * 70)

In [ ]:
# ============================================================
# 🚀 START GRADIO SERVER
# ============================================================

import threading
import time
from pyngrok import ngrok

# Stop old tunnels
ngrok.kill()

def start_gradio():
    demo.launch(
        server_name="127.0.0.1",
        server_port=7860,
        share=False,
        prevent_thread_lock=True
    )

thread = threading.Thread(
    target=start_gradio,
    daemon=True
)

thread.start()

time.sleep(5)

# ============================================================
# 🌍 NGROK TUNNEL
# ============================================================

public_url = ngrok.connect(
    7860,
    "http"
)

print("=" * 70)
print("🚀 PERSONAL FINANCE AGENTIC RAG")
print("=" * 70)
print()
print("🌐 Public URL:")
print(public_url.public_url)
print()
print("🔒 Knowledge Base : Personal Finance.pdf")
print("📚 Sources        : PDF only")
print("🤖 Architecture   : Agentic RAG")
print("=" * 70)